In [ ]:
!pip install -q segmentation-models-pytorch gdown

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 5.3 MB/s eta 0:00:00


In [ ]:
import gdown

file_id = "12dNsuo452p7BJN9SjItWJCJtZDCOldHe"
output = "/kaggle/working/test.zip"

gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=12dNsuo452p7BJN9SjItWJCJtZDCOldHe
From (redirected): https://drive.google.com/uc?id=12dNsuo452p7BJN9SjItWJCJtZDCOldHe&confirm=t&uuid=d72cf5be-195e-4577-8d5d-d373f8100fec
To: /kaggle/working/test.zip
100%|██████████| 1.16G/1.16G [00:04<00:00, 255MB/s]


'/kaggle/working/test.zip'

In [ ]:
!unzip -q /kaggle/working/test.zip -d /kaggle/working/data

In [ ]:
import gdown

file_id = "1MfTR8ljaI6SXmx453vUnu0VObg7kc22u"
output = "/kaggle/working/model.pth"

gdown.download(id=file_id, output=output, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1MfTR8ljaI6SXmx453vUnu0VObg7kc22u
From (redirected): https://drive.google.com/uc?id=1MfTR8ljaI6SXmx453vUnu0VObg7kc22u&confirm=t&uuid=f85a6865-c03a-4472-8c89-904a86ea0957
To: /kaggle/working/model.pth
100%|██████████| 294M/294M [00:03<00:00, 77.4MB/s] 


'/kaggle/working/model.pth'

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import cv2
from pathlib import Path
from tqdm import tqdm
import json
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
import segmentation_models_pytorch as smp

print("="*70)
print("DUALITY AI CHALLENGE - SEMANTIC SEGMENTATION TEST")
print("="*70)

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

DUALITY AI CHALLENGE - SEMANTIC SEGMENTATION TEST


In [ ]:
class TestConfig:
    # Paths
    CHECKPOINT_PATH = "/kaggle/working/model.pth"
    TEST_IMG_DIR = "/kaggle/working/data/test/Color_Images"
    TEST_MASK_DIR = "/kaggle/working/data/test/Segmentation"
    OUTPUT_DIR = "/kaggle/working/test_results"

    # Model
    ENCODER = "resnet34"
    NUM_CLASSES = 10
    IMG_HEIGHT = 544
    IMG_WIDTH = 960
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Test settings
    BATCH_SIZE = 8
    NUM_WORKERS = 2
    VISUALIZE_SAMPLES = 10  # Number of samples to visualize

    # Class info
    CLASS_MAPPING = {
        100: 0,    # Trees
        200: 1,    # Lush Bushes
        300: 2,    # Dry Grass
        500: 3,    # Dry Bushes
        550: 4,    # Ground Clutter
        600: 5,    # Flowers
        700: 6,    # Logs
        800: 7,    # Rocks
        7100: 8,   # Landscape
        10000: 9,  # Sky
    }

    CLASS_NAMES = [
        "Trees", "Lush Bushes", "Dry Grass", "Dry Bushes",
        "Ground Clutter", "Flowers", "Logs", "Rocks",
        "Landscape", "Sky"
    ]

    # Color palette for visualization (RGB)
    CLASS_COLORS = [
        [34, 139, 34],      # Trees - Forest Green
        [50, 205, 50],      # Lush Bushes - Lime Green
        [154, 205, 50],     # Dry Grass - Yellow Green
        [139, 69, 19],      # Dry Bushes - Saddle Brown
        [160, 82, 45],      # Ground Clutter - Sienna
        [255, 182, 193],    # Flowers - Light Pink
        [139, 90, 43],      # Logs - Brown
        [128, 128, 128],    # Rocks - Gray
        [210, 180, 140],    # Landscape - Tan
        [135, 206, 235],    # Sky - Sky Blue
    ]

# Create output directories
Path(TestConfig.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(TestConfig.OUTPUT_DIR + "/predictions").mkdir(parents=True, exist_ok=True)
Path(TestConfig.OUTPUT_DIR + "/visualizations").mkdir(parents=True, exist_ok=True)

print(f"\n✓ Configuration loaded")
print(f"✓ Test images: {TestConfig.TEST_IMG_DIR}")
print(f"✓ Output: {TestConfig.OUTPUT_DIR}")
print(f"✓ Device: {TestConfig.DEVICE}")


✓ Configuration loaded
✓ Test images: /kaggle/working/data/test/Color_Images
✓ Output: /kaggle/working/test_results
✓ Device: cuda


In [ ]:
class TestDataset(torch.utils.data.Dataset):
    """Test dataset for inference"""
    def __init__(self, image_dir):
        self.image_dir = Path(image_dir)
        self.image_paths = sorted(list(self.image_dir.glob("*.png")))
        if len(self.image_paths) == 0:
            self.image_paths = sorted(list(self.image_dir.glob("*.jpg")))

        print(f"\n✓ Found {len(self.image_paths)} test images")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]

        # Load image
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        original_image = image.copy()

        # Resize
        image = cv2.resize(image, (TestConfig.IMG_WIDTH, TestConfig.IMG_HEIGHT))

        # Normalize
        image = image.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        image = (image - mean) / std

        # To tensor
        image = torch.from_numpy(image.transpose(2, 0, 1)).float()

        return image, original_image, img_path.name

# Create test dataset
test_dataset = TestDataset(TestConfig.TEST_IMG_DIR)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=TestConfig.BATCH_SIZE,
    shuffle=False,
    num_workers=TestConfig.NUM_WORKERS,
    pin_memory=True
)

print(f"✓ Test dataloader created: {len(test_loader)} batches")


✓ Found 1002 test images
✓ Test dataloader created: 126 batches


In [ ]:
print(f"\n{'='*70}")
print("LOADING MODEL")
print(f"{'='*70}")

def load_model_safely(checkpoint_path, device):
    """Safely load model checkpoint with compatibility for different PyTorch versions"""

    # Create model architecture
    model = smp.Unet(
        encoder_name=TestConfig.ENCODER,
        encoder_weights=None,
        in_channels=3,
        classes=TestConfig.NUM_CLASSES,
        activation=None
    )

    print(f"Loading checkpoint: {checkpoint_path}")

    checkpoint = None

    try:
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        print("✓ Loaded with weights_only=False")
    except TypeError:
        # Method 2: Older PyTorch versions
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            print("✓ Loaded with default settings")
        except Exception as e:
            print(f"✗ Failed to load: {e}")
            raise
    except Exception as e:
        print(f"✗ Failed to load: {e}")
        raise

    # Extract state dict
    if checkpoint is None:
        raise ValueError("Failed to load checkpoint")

    if 'model_state_dict' in checkpoint:
        state_dict = checkpoint['model_state_dict']
        print("✓ Extracted model_state_dict from checkpoint")
    else:
        state_dict = checkpoint
        print("✓ Using checkpoint directly as state_dict")

    from collections import OrderedDict
    new_state_dict = OrderedDict()
    for k, v in state_dict.items():
        name = k.replace('module.', '')
        new_state_dict[name] = v

    # Load weights
    try:
        model.load_state_dict(new_state_dict, strict=True)
        print("✓ Weights loaded successfully (strict mode)")
    except RuntimeError as e:
        print(f"⚠️  Strict loading failed, trying non-strict mode")
        model.load_state_dict(new_state_dict, strict=False)
        print("✓ Weights loaded (non-strict mode)")

    model = model.to(device)
    model.eval()

    # Print model info
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"\n📊 Model Information:")
    print(f"  • Architecture: UNet-{TestConfig.ENCODER}")
    print(f"  • Total parameters: {total_params:,}")
    print(f"  • Trainable parameters: {trainable_params:,}")
    print(f"  • Model size: {total_params * 4 / (1024**2):.2f} MB (FP32)")

    # Print training metrics if available
    if 'metrics' in checkpoint:
        print(f"\n📈 Training Metrics:")
        metrics = checkpoint['metrics']
        for key in ['val_iou', 'val_acc', 'train_iou', 'train_acc']:
            if key in metrics and isinstance(metrics[key], float):
                print(f"  • {key}: {metrics[key]:.4f}")

    if 'epoch' in checkpoint:
        print(f"  • Trained epochs: {checkpoint['epoch'] + 1}")

    if 'config' in checkpoint:
        print(f"\n⚙️  Checkpoint Config:")
        for key, value in checkpoint['config'].items():
            print(f"  • {key}: {value}")

    return model, checkpoint

# Load model
model, checkpoint = load_model_safely(TestConfig.CHECKPOINT_PATH, TestConfig.DEVICE)

print("="*70)


LOADING MODEL
Loading checkpoint: /kaggle/working/model.pth
✓ Loaded with weights_only=False
✓ Extracted model_state_dict from checkpoint
✓ Weights loaded successfully (strict mode)

📊 Model Information:
  • Architecture: UNet-resnet34
  • Total parameters: 24,437,674
  • Trainable parameters: 24,437,674
  • Model size: 93.22 MB (FP32)

📈 Training Metrics:
  • val_iou: 0.6032
  • val_acc: 0.8802
  • train_iou: 0.6023
  • train_acc: 0.8547
  • Trained epochs: 99

⚙️  Checkpoint Config:
  • encoder: resnet34
  • num_classes: 10
  • img_size: (544, 960)
  • run_name: unet-resnet34-multi-gpu-20260131-070945


In [ ]:
print(f"\n{'='*70}")
print("QUICK LATENCY CHECK")
print(f"{'='*70}\n")

import time

# Warmup
print("Warming up...")
dummy_input = torch.randn(1, 3, TestConfig.IMG_HEIGHT, TestConfig.IMG_WIDTH).to(TestConfig.DEVICE)
with torch.no_grad():
    for _ in range(10):
        _ = model(dummy_input)

if TestConfig.DEVICE == 'cuda':
    torch.cuda.synchronize()

# Measure
print("Measuring latency (100 runs)...")
times = []
with torch.no_grad():
    for _ in tqdm(range(100)):
        if TestConfig.DEVICE == 'cuda':
            torch.cuda.synchronize()

        start = time.perf_counter()
        _ = model(dummy_input)

        if TestConfig.DEVICE == 'cuda':
            torch.cuda.synchronize()

        times.append(time.perf_counter() - start)

avg_time_ms = np.mean(times) * 1000
fps = 1000 / avg_time_ms

print(f"\n⚡ Results:")
print(f"  • Average latency: {avg_time_ms:.2f} ms")
print(f"  • FPS: {fps:.2f}")
print(f"  • Real-time capable (30 FPS): {'✅ Yes' if fps >= 30 else '❌ No'}")

if TestConfig.DEVICE == 'cuda':
    mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print(f"  • GPU memory: {mem_mb:.2f} MB")

print("="*70 + "\n")


QUICK LATENCY CHECK

Warming up...
Measuring latency (100 runs)...


100%|██████████| 100/100 [00:01<00:00, 54.37it/s]


⚡ Results:
  • Average latency: 18.22 ms
  • FPS: 54.89
  • Real-time capable (30 FPS): ✅ Yes
  • GPU memory: 575.03 MB



In [ ]:
print(f"\n{'='*70}")
print("RUNNING INFERENCE")
print(f"{'='*70}\n")

predictions = []
original_images = []
filenames = []

with torch.no_grad():
    for batch_idx, (images, orig_imgs, names) in enumerate(tqdm(test_loader, desc="Inference")):
        images = images.to(TestConfig.DEVICE)

        # Predict
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()

        # Store results
        for pred, orig_img, name in zip(preds, orig_imgs, names):
            predictions.append(pred)
            original_images.append(orig_img)
            filenames.append(name)

print(f"\n✓ Inference complete: {len(predictions)} predictions generated")


RUNNING INFERENCE



Inference: 100%|██████████| 126/126 [00:29<00:00,  4.23it/s]


✓ Inference complete: 1002 predictions generated


In [ ]:
print(f"\n{'='*70}")
print("CALCULATING STATISTICS")
print(f"{'='*70}\n")

# Calculate class distribution in predictions
class_pixel_counts = {i: 0 for i in range(TestConfig.NUM_CLASSES)}
total_pixels = 0

for pred in predictions:
    unique, counts = np.unique(pred, return_counts=True)
    for cls, count in zip(unique, counts):
        class_pixel_counts[cls] += count
        total_pixels += count

# Print statistics
print("Class Distribution in Predictions:")
print("-" * 70)
for class_idx, class_name in enumerate(TestConfig.CLASS_NAMES):
    pixel_count = class_pixel_counts[class_idx]
    percentage = (pixel_count / total_pixels) * 100
    print(f"{class_name:15s}: {pixel_count:12,} pixels ({percentage:5.2f}%)")

# Save statistics
stats = {
    "test_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "num_test_images": len(predictions),
    "model_encoder": TestConfig.ENCODER,
    "input_size": f"{TestConfig.IMG_HEIGHT}x{TestConfig.IMG_WIDTH}",
    "class_distribution": {
        name: {
            "pixels": int(class_pixel_counts[idx]),
            "percentage": float((class_pixel_counts[idx] / total_pixels) * 100)
        }
        for idx, name in enumerate(TestConfig.CLASS_NAMES)
    }
}

stats_path = Path(TestConfig.OUTPUT_DIR) / "test_statistics.json"
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)

print(f"\n✓ Statistics saved to: {stats_path}")


CALCULATING STATISTICS

Class Distribution in Predictions:
----------------------------------------------------------------------
Trees          :    1,194,134 pixels ( 0.23%)
Lush Bushes    :    9,594,661 pixels ( 1.83%)
Dry Grass      :  109,178,434 pixels (20.86%)
Dry Bushes     :    8,970,029 pixels ( 1.71%)
Ground Clutter :   67,394,358 pixels (12.88%)
Flowers        :    5,913,859 pixels ( 1.13%)
Logs           :       44,934 pixels ( 0.01%)
Rocks          :    5,868,143 pixels ( 1.12%)
Landscape      :  219,748,409 pixels (41.99%)
Sky            :   95,377,519 pixels (18.23%)

✓ Statistics saved to: /kaggle/working/test_results/test_statistics.json


In [ ]:
print(f"\n{'='*70}")
print("CALCULATING TEST METRICS")
print(f"{'='*70}\n")

test_mask_dir = Path(TestConfig.TEST_MASK_DIR)
if test_mask_dir.exists():
    print(f"✓ Ground truth found: {test_mask_dir}")

    # Load GT masks with proper resizing
    gt_masks = []
    valid_pairs = []

    print("Loading and preprocessing ground truth...")
    for idx, filename in enumerate(tqdm(filenames, desc="Loading GT")):
        mask_path = test_mask_dir / filename
        if mask_path.exists():
            gt_mask = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
            pred = predictions[idx]

            # FIX 1: Resize GT from 540x960 to 544x960 to match predictions
            if gt_mask.shape != pred.shape:
                gt_mask = cv2.resize(gt_mask, (pred.shape[1], pred.shape[0]),
                                    interpolation=cv2.INTER_NEAREST)

            # FIX 2: Remap GT to class indices
            gt_remapped = np.zeros(pred.shape, dtype=np.int64)
            for old_id, new_id in TestConfig.CLASS_MAPPING.items():
                gt_remapped[gt_mask == old_id] = new_id

            gt_masks.append(gt_remapped)
            valid_pairs.append(idx)

    print(f"✓ Loaded {len(valid_pairs)} ground truth masks\n")

    # Calculate IoU with proper handling of missing classes
    def calculate_iou_per_class(pred, target, num_classes):
        """Calculate IoU for each class"""
        ious = np.full(num_classes, np.nan)

        for cls in range(num_classes):
            pred_mask = (pred == cls)
            target_mask = (target == cls)

            # Only calculate if class exists in GT
            if target_mask.sum() > 0:
                intersection = np.logical_and(pred_mask, target_mask).sum()
                union = np.logical_or(pred_mask, target_mask).sum()
                ious[cls] = intersection / union if union > 0 else 0.0
            else:
                # Class doesn't exist in GT - mark as NaN
                ious[cls] = np.nan

        return ious

    # Compute metrics
    all_ious = []
    all_accuracies = []
    class_totals = np.zeros(TestConfig.NUM_CLASSES)  # Count images with each class

    print("Calculating metrics...")
    for idx in tqdm(valid_pairs):
        pred = predictions[idx]
        gt = gt_masks[idx]

        # IoU per class
        iou = calculate_iou_per_class(pred, gt, TestConfig.NUM_CLASSES)
        all_ious.append(iou)

        # Track which classes are present
        for cls in range(TestConfig.NUM_CLASSES):
            if (gt == cls).sum() > 0:
                class_totals[cls] += 1

        # Pixel accuracy
        acc = (pred == gt).sum() / pred.size
        all_accuracies.append(acc)

    # Calculate mean metrics
    all_ious = np.array(all_ious)
    mean_ious = np.nanmean(all_ious, axis=0)

    # FIX 3: Calculate mean IoU only for classes that exist in test set
    valid_class_ious = mean_ious[~np.isnan(mean_ious)]
    mean_iou = np.mean(valid_class_ious) if len(valid_class_ious) > 0 else 0.0
    mean_accuracy = np.mean(all_accuracies)

    # Print results
    print("\n" + "="*70)
    print("TEST SET METRICS")
    print("="*70)
    print(f"\n📊 Overall Metrics:")
    print(f"  • Mean IoU (classes in test): {mean_iou:.4f}")
    print(f"  • Pixel Accuracy:             {mean_accuracy:.4f}")
    print(f"  • Test Images:                {len(valid_pairs)}")
    print(f"  • Classes in test set:        {int(np.sum(class_totals > 0))}/10")

    print(f"\n📈 Per-Class Metrics:")
    print("-" * 95)
    print(f"{'Class':<15} {'IoU':>8} {'In Test?':>10} {'# Images':>10} {'% Present':>10} {'Status':>15}")
    print("-" * 95)

    for i, class_name in enumerate(TestConfig.CLASS_NAMES):
        iou = mean_ious[i]
        count = int(class_totals[i])
        pct = (count / len(valid_pairs)) * 100 if count > 0 else 0

        if np.isnan(iou) or count == 0:
            iou_str = "N/A"
            in_test = "✗ No"
            status = "Not in test"
        else:
            iou_str = f"{iou:.4f}"
            in_test = "✓ Yes"
            status = "Present in test"

        print(f"{class_name:<15} {iou_str:>8} {in_test:>10} {count:>10} {pct:>9.1f}% {status:>15}")

    print("-" * 95)

    # Recalculate with only classes present in test
    print(f"\n📊 Adjusted Metrics (only classes in test set):")
    present_classes = [i for i, name in enumerate(TestConfig.CLASS_NAMES) if class_totals[i] > 0]
    print(f"  • Classes evaluated: {[TestConfig.CLASS_NAMES[i] for i in present_classes]}")
    print(f"  • Mean IoU (adjusted): {mean_iou:.4f}")

    # Save metrics
    test_metrics = {
        "test_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "num_test_images": len(valid_pairs),
        "overall_metrics": {
            "mean_iou_all_classes": float(np.nanmean(mean_ious)),
            "mean_iou_present_classes": float(mean_iou),
            "pixel_accuracy": float(mean_accuracy),
        },
        "per_class_metrics": {
            name: {
                "iou": float(mean_ious[i]) if not np.isnan(mean_ious[i]) else None,
                "present_in_test": bool(class_totals[i] > 0),
                "num_images": int(class_totals[i]),
                "percentage": float((class_totals[i] / len(valid_pairs)) * 100),
            }
            for i, name in enumerate(TestConfig.CLASS_NAMES)
        },
        "notes": {
            "missing_classes": [TestConfig.CLASS_NAMES[i] for i in range(TestConfig.NUM_CLASSES) if class_totals[i] == 0],
            "size_correction": "GT resized from 540x960 to 544x960 to match predictions"
        }
    }

    metrics_path = Path(TestConfig.OUTPUT_DIR) / "test_metrics.json"
    with open(metrics_path, 'w') as f:
        json.dump(test_metrics, f, indent=2)

    print(f"\n✓ Metrics saved to: {metrics_path}")
    print("="*70 + "\n")

else:
    print(f"⚠️  Ground truth not found")
    print("="*70 + "\n")


CALCULATING TEST METRICS

✓ Ground truth found: /kaggle/working/data/test/Segmentation
Loading and preprocessing ground truth...


Loading GT: 100%|██████████| 1002/1002 [00:12<00:00, 82.81it/s]


✓ Loaded 1002 ground truth masks

Calculating metrics...


100%|██████████| 1002/1002 [00:19<00:00, 50.47it/s]


TEST SET METRICS

📊 Overall Metrics:
  • Mean IoU (classes in test): 0.4151
  • Pixel Accuracy:             0.6578
  • Test Images:                1002
  • Classes in test set:        7/10

📈 Per-Class Metrics:
-----------------------------------------------------------------------------------------------
Class                IoU   In Test?   # Images  % Present          Status
-----------------------------------------------------------------------------------------------
Trees             0.4780      ✓ Yes        986      98.4% Present in test
Lush Bushes       0.0004      ✓ Yes        668      66.7% Present in test
Dry Grass         0.4493      ✓ Yes       1002     100.0% Present in test
Dry Bushes        0.2946      ✓ Yes       1002     100.0% Present in test
Ground Clutter       N/A       ✗ No          0       0.0%     Not in test
Flowers              N/A       ✗ No          0       0.0%     Not in test
Logs                 N/A       ✗ No          0       0.0%     Not in test
Rock


/tmp/ipykernel_55/2714588795.py:80: RuntimeWarning: Mean of empty slice
  mean_ious = np.nanmean(all_ious, axis=0)


In [ ]:
print(f"\n{'='*70}")
print("SAVING PREDICTIONS")
print(f"{'='*70}\n")

# Reverse mapping (class index -> original ID)
REVERSE_MAPPING = {v: k for k, v in TestConfig.CLASS_MAPPING.items()}

for pred, filename in tqdm(zip(predictions, filenames), total=len(predictions), desc="Saving masks"):
    # Convert class indices back to original IDs
    pred_mask = np.zeros_like(pred, dtype=np.uint16)
    for class_idx, original_id in REVERSE_MAPPING.items():
        pred_mask[pred == class_idx] = original_id

    # Save as 16-bit PNG
    save_path = Path(TestConfig.OUTPUT_DIR) / "predictions" / filename
    cv2.imwrite(str(save_path), pred_mask)

print(f"✓ Predictions saved to: {TestConfig.OUTPUT_DIR}/predictions/")


SAVING PREDICTIONS



Saving masks: 100%|██████████| 1002/1002 [00:10<00:00, 96.53it/s]

✓ Predictions saved to: /kaggle/working/test_results/predictions/


In [ ]:
print(f"\n{'='*70}")
print("CREATING VISUALIZATIONS")
print(f"{'='*70}\n")

def mask_to_rgb(mask):
    """Convert class mask to RGB image"""
    h, w = mask.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)

    for class_idx, color in enumerate(TestConfig.CLASS_COLORS):
        rgb[mask == class_idx] = color

    return rgb

def overlay_mask(image, mask, alpha=0.5):
    """Overlay segmentation mask on image"""
    # Convert to numpy if needed
    if not isinstance(image, np.ndarray):
        image = np.array(image)

    mask_rgb = mask_to_rgb(mask)

    # Resize mask to match original image size
    if image.shape[:2] != mask.shape:
        mask_rgb = cv2.resize(mask_rgb, (image.shape[1], image.shape[0]))

    # Ensure same dtype
    image = image.astype(np.uint8)
    mask_rgb = mask_rgb.astype(np.uint8)

    overlay = cv2.addWeighted(image, 1-alpha, mask_rgb, alpha, 0)
    return overlay

# Visualize random samples
np.random.seed(42)
viz_indices = np.random.choice(len(predictions), min(TestConfig.VISUALIZE_SAMPLES, len(predictions)), replace=False)

for idx in tqdm(viz_indices, desc="Creating visualizations"):
    pred = predictions[idx]
    orig_img = original_images[idx]
    filename = filenames[idx]

    # Resize prediction to original size
    pred_resized = cv2.resize(pred.astype(np.uint8), (orig_img.shape[1], orig_img.shape[0]), interpolation=cv2.INTER_NEAREST)

    # Create figure
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Original image
    axes[0].imshow(orig_img)
    axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
    axes[0].axis('off')

    # Segmentation mask
    mask_rgb = mask_to_rgb(pred_resized)
    axes[1].imshow(mask_rgb)
    axes[1].set_title("Segmentation Mask", fontsize=14, fontweight='bold')
    axes[1].axis('off')

    # Overlay
    overlay = overlay_mask(orig_img, pred_resized, alpha=0.4)
    axes[2].imshow(overlay)
    axes[2].set_title("Overlay (40% transparency)", fontsize=14, fontweight='bold')
    axes[2].axis('off')

    # Add legend
    patches = [mpatches.Patch(color=np.array(color)/255., label=name)
               for color, name in zip(TestConfig.CLASS_COLORS, TestConfig.CLASS_NAMES)]
    fig.legend(handles=patches, loc='lower center', ncol=5, fontsize=10, frameon=True)

    plt.suptitle(f"Test Sample: {filename}", fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout(rect=[0, 0.08, 1, 0.96])

    # Save
    save_path = Path(TestConfig.OUTPUT_DIR) / "visualizations" / f"viz_{filename}"
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

print(f"✓ Visualizations saved to: {TestConfig.OUTPUT_DIR}/visualizations/")


CREATING VISUALIZATIONS



Creating visualizations: 100%|██████████| 10/10 [00:09<00:00,  1.04it/s]

✓ Visualizations saved to: /kaggle/working/test_results/visualizations/


In [ ]:
print(f"\n{'='*70}")
print("CREATING SUMMARY VISUALIZATION")
print(f"{'='*70}\n")

# Class distribution bar chart
fig, ax = plt.subplots(figsize=(12, 6))

class_names = TestConfig.CLASS_NAMES
percentages = [(class_pixel_counts[i] / total_pixels) * 100 for i in range(TestConfig.NUM_CLASSES)]
colors_normalized = [np.array(color)/255. for color in TestConfig.CLASS_COLORS]

bars = ax.bar(class_names, percentages, color=colors_normalized, edgecolor='black', linewidth=1.5)

ax.set_xlabel('Class', fontsize=12, fontweight='bold')
ax.set_ylabel('Percentage of Pixels (%)', fontsize=12, fontweight='bold')
ax.set_title('Class Distribution in Test Set Predictions', fontsize=14, fontweight='bold')
ax.set_ylim(0, max(percentages) * 1.1)
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', alpha=0.3, linestyle='--')

# Add value labels on bars
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
summary_path = Path(TestConfig.OUTPUT_DIR) / "class_distribution_summary.png"
plt.savefig(summary_path, dpi=150, bbox_inches='tight')
plt.close()

print(f"✓ Summary chart saved to: {summary_path}")


CREATING SUMMARY VISUALIZATION

✓ Summary chart saved to: /kaggle/working/test_results/class_distribution_summary.png


In [ ]:
!zip -r test_results.zip test_results

  adding: test_results/ (stored 0%)
  adding: test_results/predictions/ (stored 0%)
  adding: test_results/predictions/0000966.png (deflated 22%)
  adding: test_results/predictions/0000298.png (deflated 22%)
  adding: test_results/predictions/0000601.png (deflated 21%)
  adding: test_results/predictions/0001010.png (deflated 20%)
  adding: test_results/predictions/0001048.png (deflated 20%)
  adding: test_results/predictions/0000203.png (deflated 21%)
  adding: test_results/predictions/0000946.png (deflated 24%)
  adding: test_results/predictions/0000578.png (deflated 21%)
  adding: test_results/predictions/0000684.png (deflated 22%)
  adding: test_results/predictions/0001050.png (deflated 21%)
  adding: test_results/predictions/0000372.png (deflated 23%)
  adding: test_results/predictions/0001004.png (deflated 23%)
  adding: test_results/predictions/0000572.png (deflated 25%)
  adding: test_results/predictions/0000402.png (deflated 23%)
  adding: test_results/predictions/0000071.png (